[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umangbagadi03/build-your-own-coding-agent/blob/main/notebooks/20_agent_v4.ipynb)

# 20. Agent v4

**Part 4: Reliability** · about 30 minutes

Part 4 was about the difference between an agent that works in a demo and one you can rely on. This notebook puts the pieces together and then tests the result the hard way: the whole eval suite from notebook 18, run over a network where **one request in four fails**.

| | Agent v3 and earlier | Agent v4 |
|---|---|---|
| Failed API call | The run crashes | Retries with backoff (notebook 16) |
| Long outage | All work lost | Checkpoint after each step, resume later (notebook 16) |
| What happened? | A printed trace | A JSON Lines trace for every run (notebook 17) |
| Is it any good? | One run, judged by eye | An eval suite with hidden tests (notebook 18) |
| Cost | Unknown | Measured per run, with a budget (notebook 19) |

In [1]:
import pathlib, urllib.request

if not pathlib.Path("llm_setup.py").exists():  # Colab opens one notebook at a time, so fetch the helper
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/umangbagadi03/build-your-own-coding-agent/main/notebooks/llm_setup.py", "llm_setup.py"
    )

from llm_setup import connect

llm = connect()

OFFLINE MODE: a scripted local server, not a real model.
The HTTP calls are real. The answers are canned for these lessons.
Token counts are rough estimates and timings are simulated.
Set GROQ_API_KEY (free tier) or LLM_PROVIDER=ollama to use a real model.


In [2]:
import json
import os
import random
import re
import shlex
import shutil
import subprocess
import sys
import tempfile
import threading
import time
import uuid
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

import requests

headers = {"Content-Type": "application/json"}
if llm.api_key:
    headers["Authorization"] = f"Bearer {llm.api_key}"

## 1. The tasks, the tools and the graders

These are the four tasks, the shell-free toolbox and the graders from notebook 18, unchanged.

In [3]:
TASKS = [
    {
        "id": "empty-file",
        "main": "wordcount.py",
        "report": "Bug report: `python wordcount.py empty.txt` crashes on an empty file.",
        "files": {
            "wordcount.py": '"""Print word statistics for a text file."""\n\nimport sys\n\n\ndef stats(text):\n    words = text.split()\n    total_letters = sum(len(word) for word in words)\n    return {"words": len(words), "average_length": round(total_letters / len(words), 2)}\n\n\ndef main(path):\n    with open(path) as f:\n        result = stats(f.read())\n    print(f"{result[\'words\']} words, average length {result[\'average_length\']}")\n\n\nif __name__ == "__main__":\n    main(sys.argv[1])\n',
            "test_wordcount.py": 'import unittest\n\nfrom wordcount import stats\n\n\nclass StatsTests(unittest.TestCase):\n    def test_counts_words(self):\n        self.assertEqual(stats("one two three")["words"], 3)\n',
            "empty.txt": "",
        },
        "hidden_tests": 'import unittest\n\nfrom wordcount import stats\n\n\nclass Hidden(unittest.TestCase):\n    def test_empty(self):\n        self.assertEqual(stats(""), {"words": 0, "average_length": 0.0})\n\n    def test_only_whitespace(self):\n        self.assertEqual(stats("  \\n  ")["words"], 0)\n\n    def test_normal_text_unchanged(self):\n        self.assertEqual(stats("ab abcd")["average_length"], 3.0)\n',
    },
    {
        "id": "boiling-point",
        "main": "convert.py",
        "report": "Bug report: `python convert.py 100` prints 203.0, but water boils at 212 F.",
        "files": {
            "convert.py": '"""Convert Celsius to Fahrenheit."""\n\nimport sys\n\n\ndef to_fahrenheit(celsius):\n    return celsius * 9 / 5 + 23\n\n\nif __name__ == "__main__":\n    print(to_fahrenheit(float(sys.argv[1])))\n',
            "test_convert.py": 'import unittest\n\nfrom convert import to_fahrenheit\n\n\nclass ConvertTests(unittest.TestCase):\n    def test_returns_a_number(self):\n        self.assertIsInstance(to_fahrenheit(10.0), float)\n',
        },
        "hidden_tests": 'import unittest\n\nfrom convert import to_fahrenheit\n\n\nclass Hidden(unittest.TestCase):\n    def test_boiling(self):\n        self.assertEqual(to_fahrenheit(100), 212)\n\n    def test_freezing(self):\n        self.assertEqual(to_fahrenheit(0), 32)\n\n    def test_minus_forty(self):\n        self.assertEqual(to_fahrenheit(-40), -40)\n',
    },
    {
        "id": "double-space",
        "main": "names.py",
        "report": "Bug report: `python names.py 'Asha  Verma'` (two spaces between the names) crashes.",
        "files": {
            "names.py": '"""Turn a full name into initials."""\n\nimport sys\n\n\ndef initials(full_name):\n    """\'Asha Rani Verma\' becomes \'A.R.V.\'"""\n    return ".".join(part[0].upper() for part in full_name.split(" ")) + "."\n\n\nif __name__ == "__main__":\n    print(initials(sys.argv[1]))\n',
            "test_names.py": 'import unittest\n\nfrom names import initials\n\n\nclass NameTests(unittest.TestCase):\n    def test_three_names(self):\n        self.assertEqual(initials("Asha Rani Verma"), "A.R.V.")\n',
        },
        "hidden_tests": 'import unittest\n\nfrom names import initials\n\n\nclass Hidden(unittest.TestCase):\n    def test_double_space(self):\n        self.assertEqual(initials("Asha  Verma"), "A.V.")\n\n    def test_outer_spaces(self):\n        self.assertEqual(initials(" Asha Verma "), "A.V.")\n\n    def test_lower_case(self):\n        self.assertEqual(initials("asha rani verma"), "A.R.V.")\n',
    },
    {
        "id": "coupon",
        "main": "discount.py",
        "report": "Bug report: `python discount.py 200 10` prints 190.0. A 10% coupon on Rs 200 should leave Rs 180.",
        "files": {
            "discount.py": '"""Coupon maths for the shop."""\n\nimport sys\n\n\ndef apply_coupon(price, percent):\n    """Price after a percent-off coupon."""\n    return round(price - percent, 2)\n\n\nif __name__ == "__main__":\n    print(apply_coupon(float(sys.argv[1]), float(sys.argv[2])))\n',
            "test_discount.py": 'import unittest\n\nfrom discount import apply_coupon\n\n\nclass CouponTests(unittest.TestCase):\n    def test_no_coupon(self):\n        self.assertEqual(apply_coupon(100, 0), 100)\n',
        },
        "hidden_tests": 'import unittest\n\nfrom discount import apply_coupon\n\n\nclass Hidden(unittest.TestCase):\n    def test_report_example(self):\n        self.assertEqual(apply_coupon(200, 10), 180)\n\n    def test_other_price(self):\n        self.assertEqual(apply_coupon(500, 10), 450)\n\n    def test_half_off(self):\n        self.assertEqual(apply_coupon(80, 50), 40)\n',
    },
]


def make_tools(workspace):
    def safe(path):
        full = (workspace / path).resolve()
        if not full.is_relative_to(workspace):
            raise PermissionError(f"{path!r} is outside the workspace")
        return full

    def run_command(command):
        try:
            words = shlex.split(command)
        except ValueError as error:
            return f"Error: couldn't parse the command ({error})"
        is_script = len(words) >= 2 and words[0] in ("python", "python3") and words[1].endswith(".py")
        is_tests = words[:3] in (["python", "-m", "unittest"], ["python3", "-m", "unittest"])
        if not (is_script or is_tests):
            return "Denied: only `python <script>.py ...` and `python -m unittest ...` are allowed here."
        env = {k: os.environ[k] for k in ("PATH", "SYSTEMROOT") if k in os.environ}
        try:
            done = subprocess.run([sys.executable, "-B", *words[1:]], cwd=workspace, env=env,
                                  capture_output=True, text=True, timeout=30)
        except subprocess.TimeoutExpired:
            return "stopped: took longer than 30 seconds"
        return f"exit code: {done.returncode}\n" + (done.stdout + done.stderr).replace(str(workspace), ".")[-2000:]

    def read_file(path):
        return "\n".join(f"{n:>4} | {line}" for n, line in enumerate(safe(path).read_text().splitlines(), 1))

    def edit_file(path, old, new):
        text = safe(path).read_text()
        if text.count(old) != 1:
            return f"Error: old must appear exactly once; it appears {text.count(old)} times."
        safe(path).write_text(text.replace(old, new, 1))
        return f"Edited {path}."

    def create_file(path, content):
        if safe(path).exists():
            return f"Error: {path} already exists. Choose a new file name."
        safe(path).write_text(content)
        return f"Created {path}."

    return {"run_command": run_command, "read_file": read_file, "edit_file": edit_file, "create_file": create_file}


def schema(name, description, **props):
    return {"type": "function", "function": {"name": name, "description": description, "parameters": {
        "type": "object", "properties": {k: {"type": "string", "description": v} for k, v in props.items()},
        "required": list(props)}}}


TOOLS = [
    schema("run_command", "Run `python <script>.py args` or `python -m unittest` in the project. "
                          "Output starts with the exit code.", command="The command"),
    schema("read_file", "Read a file with line numbers.", path="File path"),
    schema("edit_file", "Replace text that appears exactly once in a file. No line numbers in old.",
           path="File path", old="Exact text to replace", new="Replacement"),
    schema("create_file", "Create a new file. Fails if it exists.", path="File path", content="File content"),
]

SYSTEM_PROMPT = """You fix bugs in small Python projects.
1. Reproduce the bug with the command from the report.
2. Read the code and fix the general case with edit_file. The example in the report is one input among many,
   so don't write a fix that only works for it.
3. Run the command again to confirm.
4. Add a test for the bug in a new file, then run all tests with `python -m unittest -q`.
Finish with a short summary. Only claim what you saw in tool results."""


def unittest_passes(workspace):
    done = subprocess.run([sys.executable, "-B", "-m", "unittest", "-q"], cwd=workspace, capture_output=True, text=True,
                          timeout=60, env={k: os.environ[k] for k in ("PATH", "SYSTEMROOT") if k in os.environ})
    return done.returncode == 0


def grade(task, workspace, summary):
    new_tests = [p.name for p in workspace.glob("test_*.py") if p.name not in task["files"]]
    results = {"finished": summary is not None, "added_test": bool(new_tests),
               "own_tests_pass": unittest_passes(workspace)}
    main = workspace / task["main"]
    fixed_source = main.read_text()
    main.write_text(task["files"][task["main"]])
    results["test_catches_bug"] = bool(new_tests) and not unittest_passes(workspace)
    main.write_text(fixed_source)
    (workspace / "test_hidden.py").write_text(task["hidden_tests"])
    results["hidden_tests"] = unittest_passes(workspace)
    (workspace / "test_hidden.py").unlink()
    return results

## 2. A network that keeps failing

The flaky proxy from notebook 16, with one change. Instead of following a fixed plan, it fails each request at random with a set probability, picking among the four kinds of failure. A fixed random seed makes the "random" failures the same on every run of the notebook, so results can be compared.

In [4]:
class FlakyProxy:
    def __init__(self, upstream, failure_rate, seed=7):
        self.upstream, self.failure_rate = upstream.rstrip("/"), failure_rate
        self.random, self.stats = random.Random(seed), {"forwarded": 0, "failed": 0}
        self.lock = threading.Lock()
        proxy = self

        class Handler(BaseHTTPRequestHandler):
            def log_message(self, *args):
                pass

            def do_POST(self):
                body = self.rfile.read(int(self.headers.get("Content-Length", 0)))
                with proxy.lock:
                    fail = proxy.random.random() < proxy.failure_rate
                    action = proxy.random.choice([429, 503, "hang", "drop"]) if fail else "ok"
                    proxy.stats["failed" if fail else "forwarded"] += 1
                try:
                    if action == "hang":
                        time.sleep(3)
                        return
                    if action == "drop":
                        self.connection.close()
                        return
                    if action != "ok":
                        self.send_response(action)
                        if action == 429:
                            self.send_header("Retry-After", "1")
                        self.send_header("Content-Length", "0")
                        self.end_headers()
                        return
                    forward = {k: v for k, v in self.headers.items() if k.lower() in ("authorization", "content-type")}
                    reply = requests.post(proxy.upstream + "/chat/completions", data=body, headers=forward,
                                          timeout=llm.timeout)
                    self.send_response(reply.status_code)
                    self.send_header("Content-Type", "application/json")
                    self.send_header("Content-Length", str(len(reply.content)))
                    self.end_headers()
                    self.wfile.write(reply.content)
                except (BrokenPipeError, ConnectionResetError, OSError):
                    pass

        class QuietServer(ThreadingHTTPServer):
            def handle_error(self, request, client_address):
                pass

        self.server = QuietServer(("127.0.0.1", 0), Handler)
        threading.Thread(target=self.server.serve_forever, daemon=True).start()
        self.url = f"http://127.0.0.1:{self.server.server_address[1]}/v1/chat/completions"

## 3. Agent v3-style: no retries

First, a baseline. This is the loop every agent in the course used up to notebook 15: one request per step, and any failure ends the run. A crashed run counts as a failed task.

In [5]:
def call_once(proxy, payload):
    response = requests.post(proxy.url, headers=headers, json=payload, timeout=2.0 if llm.offline else llm.timeout)
    response.raise_for_status()
    return response.json()


def run_task(task, call, tracer=None, budget=None, max_steps=15):
    workspace = Path(tempfile.mkdtemp(prefix=f"v4-{task['id']}-")).resolve()
    for path, text in task["files"].items():
        (workspace / path).write_text(text)
    tools = make_tools(workspace)
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": task["report"] + " Fix it and add a test in a new file."}]
    usage_total = {"input": 0, "output": 0}
    for step in range(1, max_steps + 1):
        if budget is not None and usage_total["input"] > budget:
            return workspace, None, usage_total, "over budget"
        started = time.perf_counter()
        data = call({"model": llm.model, "messages": messages, "tools": TOOLS})
        message = data["choices"][0]["message"]
        usage = data.get("usage") or {}
        usage_total["input"] += usage.get("prompt_tokens", 0)
        usage_total["output"] += usage.get("completion_tokens", 0)
        if tracer:
            tracer.record("model_call", step, started, message=message, input_tokens=usage.get("prompt_tokens", 0))
        messages.append(message)
        if not message.get("tool_calls"):
            return workspace, message.get("content") or "", usage_total, "finished"
        for tool_call in message["tool_calls"]:
            started = time.perf_counter()
            name = tool_call["function"]["name"]
            try:
                result = tools[name](**json.loads(tool_call["function"]["arguments"] or "{}")) if name in tools \
                    else f"Error: unknown tool {name!r}"
            except Exception as error:
                result = f"Error: {type(error).__name__}: {error}"
            if tracer:
                tracer.record("tool_call", step, started, tool=name, result=result[:500])
            messages.append({"role": "tool", "tool_call_id": tool_call["id"], "content": result})
    return workspace, None, usage_total, "out of steps"


def run_suite(call, label, **options):
    passed, rows = 0, []
    for task in TASKS:
        try:
            workspace, summary, usage, status = run_task(task, call, **options)
            results = grade(task, workspace, summary)
            shutil.rmtree(workspace)
        except Exception as error:
            status, results, usage = f"crashed: {type(error).__name__}", {"hidden_tests": False}, {"input": 0, "output": 0}
        passed += results["hidden_tests"]
        rows.append((task["id"], status, results["hidden_tests"], usage))
        print(f"  {label}  {task['id']:<15}{'PASS' if results['hidden_tests'] else 'FAIL':<6}{status}")
    print(f"  {label}  pass rate: {passed}/{len(TASKS)}")
    return rows


flaky = FlakyProxy(llm.base_url, failure_rate=0.25)
no_retries = run_suite(lambda payload: call_once(flaky, payload), "no retries")
print("\nProxy:", flaky.stats)

  no retries  empty-file     FAIL  crashed: HTTPError
  no retries  boiling-point  FAIL  crashed: HTTPError


  no retries  double-space   FAIL  crashed: HTTPError
  no retries  coupon         FAIL  crashed: ConnectionError
  no retries  pass rate: 0/4

Proxy: {'forwarded': 2, 'failed': 4}


Same agent, same tasks, same model, and the score collapses, because a dropped connection is now just as fatal as a wrong answer. If your agent's score on a bad network day is very different from its score on a good one, your evals are partly measuring the network.

## 4. Agent v4

Now the same loop with the reliability layer around it:

- **Retries** with exponential backoff, jitter and `Retry-After` (notebook 16)
- **A trace** of every model call and tool call, one JSON Lines file per run (notebook 17)
- **A token budget** per task (notebook 19). A run stops once it has used 30,000 input tokens, which only a runaway agent would reach on these tasks.

In [6]:
RETRYABLE = {408, 409, 429, 500, 502, 503, 504}


def call_with_retries(proxy, payload, max_attempts=8, base_delay=0.25, max_delay=4.0, counter=None):
    timeout = 2.0 if llm.offline else llm.timeout
    for attempt in range(1, max_attempts + 1):
        retry_after = None
        try:
            response = requests.post(proxy.url, headers=headers, json=payload, timeout=timeout)
            if response.status_code < 400:
                return response.json()
            if response.status_code not in RETRYABLE:
                raise RuntimeError(f"HTTP {response.status_code}, not retrying")
            retry_after = response.headers.get("Retry-After")
        except (requests.Timeout, requests.ConnectionError):
            pass
        if attempt == max_attempts:
            raise RuntimeError(f"gave up after {attempt} attempts")
        if counter is not None:
            counter["retries"] += 1
        time.sleep(float(retry_after) if retry_after else
                   min(max_delay, base_delay * 2 ** (attempt - 1)) * random.uniform(0.5, 1.0))


class Tracer:
    def __init__(self, folder, name):
        self.path = Path(folder) / f"{name}-{uuid.uuid4().hex[:6]}.jsonl"
        self.started = time.perf_counter()

    def record(self, kind, step, start, **details):
        event = {"type": kind, "step": step, "start": round(start - self.started, 3),
                 "duration": round(time.perf_counter() - start, 3), **details}
        with self.path.open("a") as f:
            f.write(json.dumps(event) + "\n")


traces = Path(tempfile.mkdtemp(prefix="v4-traces-"))
counter = {"retries": 0}
flaky_again = FlakyProxy(llm.base_url, failure_rate=0.25)  # same seed, so the same failures


def v4_call(payload):
    return call_with_retries(flaky_again, payload, counter=counter)


started = time.perf_counter()
v4_rows = []
for task in TASKS:
    tracer = Tracer(traces, task["id"])
    try:
        workspace, summary, usage, status = run_task(task, v4_call, tracer=tracer, budget=30_000)
        results = grade(task, workspace, summary)
        shutil.rmtree(workspace)
    except Exception as error:
        status, results, usage = f"crashed: {error}", {"hidden_tests": False}, {"input": 0, "output": 0}
    v4_rows.append((task["id"], status, results["hidden_tests"], usage))
    print(f"  v4  {task['id']:<15}{'PASS' if results['hidden_tests'] else 'FAIL':<6}{status}")
elapsed = time.perf_counter() - started
print(f"  v4  pass rate: {sum(r[2] for r in v4_rows)}/{len(TASKS)}")
print(f"\nProxy: {flaky_again.stats}. Retries made: {counter['retries']}. Time: {elapsed:.0f}s")

  v4  empty-file     PASS  finished


  v4  boiling-point  PASS  finished


  v4  double-space   PASS  finished


  v4  coupon         FAIL  finished
  v4  pass rate: 3/4

Proxy: {'forwarded': 28, 'failed': 18}. Retries made: 18. Time: 48s


In [7]:
# Checkpoint: offline, v4 rides out every network failure and scores what the script deserves (3 of 4).
if llm.offline:
    assert all(not r[1].startswith("crashed") for r in v4_rows)
    assert sum(r[2] for r in v4_rows) == 3
    assert flaky_again.stats["failed"] > 0, "The proxy should have failed some requests."
    assert sum(r[2] for r in no_retries) < 3
print("Checkpoint passed.")

Checkpoint passed.


With the same failures from the same seed, v4 finished every task. It scored 3 out of 4 in offline mode, which is exactly what the agent deserves: the coupon task still fails on the hidden tests, because retries can make an agent survive a bad network, but they can't make it smarter.

That's the point of separating the layers. Reliability work doesn't change *what* the agent does, it makes sure the agent actually gets to do it. The eval then measures the part that's left, the agent's judgement.

## 5. The report card

Everything a team would look at before shipping a change, from one run of the suite:

In [8]:
total_in = sum(r[3]["input"] for r in v4_rows)
total_out = sum(r[3]["output"] for r in v4_rows)
example_cost = (total_in * 1.00 + total_out * 5.00) / 1_000_000  # example prices from notebook 19
trace_files = sorted(traces.glob("*.jsonl"))
events = sum(len(p.read_text().splitlines()) for p in trace_files)

print("AGENT V4 REPORT CARD")
print(f"  Pass rate (hidden tests):     {sum(r[2] for r in v4_rows)}/{len(TASKS)}")
print(f"  Runs that crashed:            {sum(r[1].startswith('crashed') for r in v4_rows)}")
print(f"  Network failures absorbed:    {flaky_again.stats['failed']} (by {counter['retries']} retries)")
print(f"  Tokens:                       {total_in:,} in, {total_out:,} out"
      + (" (estimates)" if llm.offline else ""))
print(f"  Example cost for the suite:   ${example_cost:.4f} (example prices, see notebook 19)")
print(f"  Traces written:               {len(trace_files)} files, {events} events")
print(f"  Wall time:                    {elapsed:.0f}s")

AGENT V4 REPORT CARD
  Pass rate (hidden tests):     3/4
  Runs that crashed:            0
  Network failures absorbed:    18 (by 18 retries)
  Tokens:                       23,465 in, 1,325 out (estimates)
  Example cost for the suite:   $0.0301 (example prices, see notebook 19)
  Traces written:               4 files, 52 events
  Wall time:                    48s


## You finished Part 4

Your agent now survives failures, records what it does, stays within a budget and is measured on hidden tests. It's no longer something that works in a demo. It's something you can improve with evidence.

**Part 5** goes beyond one agent: subagents that handle side tasks with their own context, connecting to tools over MCP, skills the agent loads when it needs them, and hooks that let you check or block what it's about to do.

## Exercises

1. Raise `failure_rate` to 0.5. At what point does v4 start giving up, and which setting in `call_with_retries` would you change first?
2. Add checkpointing from notebook 16 to `run_task`, so a run that gives up can be resumed.
3. Run the suite on a real model with prompt A and prompt B from notebook 18, through v4, with three trials each. Does B fix the coupon task?

## Clean up

In [9]:
flaky.server.shutdown()
flaky_again.server.shutdown()
shutil.rmtree(traces)
print("Stopped the proxies and removed the traces.")

Stopped the proxies and removed the traces.
